In [1]:
import sqlite3
import yaml

PATH = r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\v1\BackEnd\Assobio\db\BaseDadosAssobio.db"

CONECTOR = sqlite3.connect (PATH)
CURSOR = CONECTOR.cursor ()

In [2]:
DATABASE = CURSOR.execute (
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    """
).fetchall ()

for x in DATABASE:
    print (x)

"""
('Assobio',)
('sqlite_sequence',)
"""

('Assobio',)
('sqlite_sequence',)


"\n('Assobio',)\n('sqlite_sequence',)\n"

In [3]:
DATABASE = CURSOR.execute (
    """
    SELECT * FROM Assobio
    """
).fetchmany (10)

for line in DATABASE:
    print (line)

(1, '2026-08-04 13:21:59.260311', 'C:\\Users\\Admin\\AppData\\Local\\Temp\\gradio\\9aadfa40e4809df056ab4cf553fab63b281735fb97cd3cc3af7ef48956e4b636\\audio3.wav', 'bom dia santo antónio muito bom dia paulinho ei da junta sim diga uma coisa então o que é que vos custa fazer aquilo aquilo o quê meu amigo até que eu já pedi tanta vez eu não sei do que é que o senhor está a falar onde é que quer falar com quem é que quer falar eu por mim falava com quem manda pois bom caro senhor mas é que a junta tem tem mais de cem funcionários nós não sabemos com quem é que o senhor quer falar com quem é que', '<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Task:** Evaluate a provided audio transcription based on three metrics: "Divertido" (Funny), "Importante" (Important), "Linguagem" (Language).\n   - **Scale:** Each metric rated 0-10.\n   - **Format:** Must follow a specific format (implied by "formato pretendido", though not explicitly shown, I should infer a clean, conci

In [22]:
MODELS = CURSOR.execute (
    """
    SELECT DISTINCT modelo
    FROM Assobio
    """
).fetchall ()

print (MODELS)

[('Bonsai 27B Q1.0',), ('Microsoft Phi 3.5 Q4.0',), ('QWEN 2.5 0.5B',), ('Mistral 7B Q4.0',), ('Microsoft Phi 4 Q4.0',), ('Amália 9B DPO Q8',)]


In [ ]:
DATABASE = CURSOR.execute (
    """
    SELECT name
    FROM pragma_table_info('Assobio')
    """
).fetchall ()

print (DATABASE) #[('id',), ('data',), ('audio',), ('transcrição',), ('auditoria',), ('modelo',), ('score',)]

COLS = {
    "id": "Identificação",
    "data": "Data da Run",
    "audio": "Caminho do Áudio",
    "transcrição": "Transcrição realizada pelo modelo ASR",
    "auditoria": "Auditoria realizada pelo SLM",
    "modelo": "Modelo que realizou a auditoria",
    "score": "Score dado pelo humano à auditoria"
}

[('id',), ('data',), ('audio',), ('transcrição',), ('auditoria',), ('modelo',), ('score',)]


In [8]:
TRANS = CURSOR.execute (
    """
    SELECT transcrição
    FROM Assobio
    """
).fetchall ()

for x in TRANS[:10]:
    print (x)

('bom dia santo antónio muito bom dia paulinho ei da junta sim diga uma coisa então o que é que vos custa fazer aquilo aquilo o quê meu amigo até que eu já pedi tanta vez eu não sei do que é que o senhor está a falar onde é que quer falar com quem é que quer falar eu por mim falava com quem manda pois bom caro senhor mas é que a junta tem tem mais de cem funcionários nós não sabemos com quem é que o senhor quer falar com quem é que',)
('bom dia santo antónio muito bom dia paulinho ei da junta sim diga uma coisa então o que é que vos custa fazer aquilo aquilo o quê meu amigo até que eu já pedi tanta vez eu não sei do que é que o senhor está a falar onde é que quer falar com quem é que quer falar eu por mim falava com quem manda pois bom caro senhor mas é que a junta tem tem mais de cem funcionários nós não sabemos com quem é que o senhor quer falar com quem é que',)
('por favor fale comigo eu estou a falar com a professora eu estava explicando à minha mulher que não paga ela diz que pag

<hr>

In [19]:
with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\code\SEMANTIC_MODEL.yaml", "r", encoding = "utf-8") as f:
    MODEL_SEM = yaml.safe_load (f)

display (MODEL_SEM["FUNCTIONS"].keys())

dict_keys(['RETURN', 'COUNT', 'MEAN', 'UNIQUE', 'JUNCTION'])

<hr>

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import outlines

from pydantic import BaseModel

import torch
import json

c:\Users\Admin\Desktop\ip\Automatic Speech Recognition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class Args (BaseModel):
    name: str

class CENAS (BaseModel):
    funct: str
    args: Args | list [Args]

In [5]:
device = "cuda" if torch.cuda.is_available () else "cpu"
path = r"C:\Users\Admin\Desktop\models\Language Models\Qwen 4B"

TOKENIZER = AutoTokenizer.from_pretrained (path)
MODEL = AutoModelForCausalLM.from_pretrained (path, device_map = device)


W0910 17:29:33.407000 22836 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 398/398 [00:01<00:00, 217.74it/s]


In [6]:
MODEL = outlines.from_transformers (MODEL, TOKENIZER)

In [20]:
mensagens = [
    {
        "role": "system", "content": 
    f"""
    És um modelo inserido dentro de uma Camada Semântica. Vais receber o prompt principal e deves de acordo com o prompt retornar JSON de acordo com o pretendido.

    O JSON pretendido é:
    'funct': Representa o nome da Função do Modelo Semântico.
    'args': Dentro de args é onde deves declarar os nomes das colunas que é suposto aplicar a função de acordo com o prompt. 
    ATENÇÃO que deves seguir o args de acordo com o formato da função. Há umas funções que aceitam mais que um args.

    <Exemplo>
    Prompt: Retorna o número de linhas da DataBase.

    Output: 'funct': 'COUNT', 'args': 'name': None
    </Exemplo>

    Vais receber o Modelo Semântico que está dividido em FUNCTIONS que representa as Funções possíveis e vais também
    receber COLUNAS que representa as colunas disponíveis na DataBase. 
    O Modelo Semântico tem o parâmetro 'args' que é fulcral ser bem interpretado. Significa que apenas aquelas colunas estão disponíveis para ser selecionadas,
    se tiver COLUNAS é porque qualquer uma pode ser selecionada, se tiver um nome em específico é porque só aquela coluna pode ser selecionada.

    MODELO SEMÂNTICO:
    {MODEL_SEM}

    Lembra-te que estás inserido num sistema completo e que deves retornar a resposta em formato JSON
    de acordo com o prompt do utilizador.
    """
    },

    {
        "role": "user", "content": "Quantas auditorias realizou o modelo Qwen ?"
    },
]

PROMPT = TOKENIZER.apply_chat_template (mensagens, tokenize = False, add_generation_prompt = True)

#print (PROMPT)

RESULTADO = MODEL (PROMPT, output_type = CENAS, max_new_tokens = 256)
print (RESULTADO)
print (type(RESULTADO))

RESULTADO = json.loads (RESULTADO)
print (RESULTADO)
print (type(RESULTADO))


{"funct": "COUNT", "args": {"name": "auditoria"} }
<class 'str'>
{'funct': 'COUNT', 'args': {'name': 'auditoria'}}
<class 'dict'>


In [18]:
def QUERY_COMPILER (semantic_query, modelo_semantic, database):

    FUNCTION = semantic_query["funct"]
    FUNCTION_ARGS = semantic_query["args"]

    SQL = None

    if modelo_semantic["FUNCTIONS"][FUNCTION]["type"] == "return_all":

        SQL = f"""
        SELECT {FUNCTION_ARGS["name"]} 
        FROM {database}
        """


    elif modelo_semantic["FUNCTIONS"][FUNCTION]["type"] == "return_len":

        SQL = f"""
        SELECT COUNT (*)
        FROM {database}
        """

    elif modelo_semantic["FUNCTIONS"][FUNCTION]["type"] == "mean":

        SQL = f"""
        SELECT AVG ({FUNCTION_ARGS["name"]})
        FROM {database}
        """

    elif modelo_semantic["FUNCTIONS"][FUNCTION]["type"] == "unique":

        SQL = f"""
        SELECT DISTINCT {FUNCTION_ARGS["name"]}
        FROM {database}
        """

    return SQL


x = QUERY_COMPILER (RESULTADO, MODEL_SEM, "Assobio")

TESTE = CURSOR.execute (x).fetchall ()

print (TESTE)

[('Bonsai 27B Q1.0',), ('Microsoft Phi 3.5 Q4.0',), ('QWEN 2.5 0.5B',), ('Mistral 7B Q4.0',), ('Microsoft Phi 4 Q4.0',), ('Amália 9B DPO Q8',)]
